# Introduction
In this Jupyter Notebook, blocks of text (like this one) provide directions, while blocks of code (like the one below) can be run to produce results. Try it out! Click in the box below, and then click the ► button in the bar above to run its code.

In [ ]:
print('Hello world!')

# Generating synthetic data
Run this cell to generate a synthetic dataset and perform other basic setup. This will take a couple minutes to run; wait until you see the message `Done!` appear below.

In [ ]:
epsilon = 5

import pyreadr, warnings, pandas as pd
from snsynth import Synthesizer
from sdv.metadata import Metadata
from sdmetrics.reports.single_table import QualityReport
from sdmetrics.visualization import get_column_plot, get_column_pair_plot
from sdmetrics.single_table import DisclosureProtection
warnings.filterwarnings("ignore", category = FutureWarning)

real = pyreadr.read_r('data/2023 Detroit Metro Area Communities Study (MST).RData')['df']
real = real.drop('debt_other_description', axis = 1)
ordinal_columns = ['birth_year', 'time_at_current_address', 'time_in_Detroit', 'anxious_past_week', 'worried_past_week', 'depressed_past_week', 'household_debt_level', 'time_unemployed', 'education', 'household_income']
categorical_columns = [col for col in real.columns if col not in ordinal_columns]

print('Running...')
synthesizer = Synthesizer.create('mst', epsilon = epsilon)
synthesizer.fit(real, ordinal_columns = ordinal_columns, categorical_columns = categorical_columns, nullable = True)
synthetic_non_categorical = synthesizer.sample(1911)
synthetic = synthetic_non_categorical.astype('category')
metadata = Metadata.detect_from_dataframe(synthetic).to_dict()['tables']['table']
print('Done!')

# Overview of differential privacy

We are using differential privacy to generate synthetic data that preserves overall correlations but guarantees a certain level of privacy protection for individuals.

<img src="DP_example.png" alt="overview of the epsilon value" style="width: 750px;"/>

# Quality Report
The SmartNoise Quality Report captures the statistical similarity between real data and synthetic data. If the synthetic and real data are statistically similar, we refer to the synthetic data as being high quality (aka high fidelity). The report runs select metrics to measure these properties and summarizes the results.

**Column Shapes:** Does the synthetic data capture the shape of each column? The shape of a column describes its overall distribution. The higher the score, the more similar the distributions of real and synthetic data. This yields a separate score for every column. The final Column Shapes score is the average of all columns.

**Column Pair Trends:** Does the synthetic data capture trends between pairs of columns? The trend between two columns describes how they vary in relation to each other, for example the correlation. The higher the score, the more the trends are alike. This yields a score between every pair of columns. The Column Pair Trends score is the average of all the scores.

Run this cell to generate the Quality Report.

In [ ]:
report = QualityReport()
report.generate(real, synthetic, metadata)

# Column Shape scores

The following visualization computes the similarity of each real column vs. its corresponding synthetic column in terms of the column shapes -- aka the *marginal distribution* or 1D histogram of the column. This metric ignores any missing values.

**(best score) 1.0:** The real data is exactly the same as the synthetic data

**(worst score) 0.0:** The real and synthetic data are as different as they can be

This example bar graph compares real and synthetic data for a single column from a fictitious dataset. Because of the differences between the frequencies of each category value, the score for this column is 0.68.

<img src="column_shapes_example.jpg" alt="example of how column shape scores are calculated" style="width: 750px;"/>

Run this cell to show the Column Shape score for each column in the dataset.

In [ ]:
report.get_visualization('Column Shapes').show()

# Comparing distributions for specific columns

Run this cell to visualize a real column against the same synthetic column. You can change `column_name` and repeat this to examine multiple columns.

In [ ]:
column_name = 'birth_year'

get_column_plot(real_data = real, synthetic_data = synthetic, column_name = column_name).show()

# Column Pair Trend scores

This metric computes the similarity of a pair of categorical columns between the real and synthetic datasets -- aka it compares 2D distributions. If there are missing values in the columns, the metric will treat them as an additional, single category.

**(best score) 1.0:** The contingency table is exactly the same between the real vs. synthetic data

**(worst score) 0.0:** The contingency table is as different as can be

These example plots compare real and synthetic data for pair of columns—country (vertical) and whether a user is subscribed (horizontal)—from a fictitious dataset. Because of the differences between the frequencies of each pair of category values, the score for this column pair is 0.92.

<img src="column_pair_trends_example.jpg" alt="example of how column pair trend scores are calculated" style="width: 750px;"/>

Run this cell to show the Column Pair Trend score for each pair of columns in the dataset.

In [ ]:
report.get_visualization('Column Pair Trends').show()

# Comparing distributions for specific column pairs

Run this cell to visualize the trends between a pair of columns for real and synthetic data. You can change `column_1_name` and `column_2_name` and repeat this to examine multiple pairs.

In [ ]:
column_1_name = 'birth_year'
column_2_name = 'household_income'

get_column_pair_plot(real_data = real, synthetic_data = synthetic, column_names = [column_1_name, column_2_name], plot_type = 'heatmap').show()

# Disclosure Protection Score

The value of epsilon you chose when you generated synthetic data provides a guaranteed level of privacy protection. However, since that protection can be hard to interpret, SmartNoise also provides the Disclosure Protection Score, another way to measure the risk associated with disclosing (aka broadly sharing) the synthetic data. It's a useful measurement if you want to know whether synthetic data is leaking patterns that pertain to sensitive information.

**The attack scenario:** If an attacker has prior knowledge about certain attributes (e.g. a  person's age and gender) *and* they are given access to the full synthetic data, would they be able to make better guesses about what they don't know (e.g. that person's political affiliation)?

This metric simulates the attack scenario using your real and synthetic data. It describes how much your synthetic data protects against the risk of disclosure as compared to a baseline of completely random data.

**(best score) 1.0:** The synthetic data provides strong disclosure protection. Sharing the synthetic data provides no more risk than sharing completely random values.

**(worst score) 0.0:** The synthetic data does not provide disclosure protection. Sharing the synthetic data divulges patterns that make it easy to guess sensitive attributes.

Scores between 0.0 and 1.0 indicate the relative risk of disclosure. For example, a score of 0.825 indicates that the synthetic data has 82.5% of the protection that random data would provide.

In the example visual, the average data safety across all rows is is 55% for synthetic data. As a baseline, random data would have a safety of 66.66% (because there is a ⅓ chance of guessing the value correctly). The overall DisclosureProtection risk is then 82.5%.

<img src="disclosure_protection_score_example.jpg" alt="example of how the disclosure protection score is calculated" style="width: 500px;"/>

In `known_column_names`, use your best judgment to list potentially identifying columns that might be known to an adversary. In `sensitive_column_names`, list columns containing sensitive information. Please feel free to add more column names to both lists.

In [ ]:
known_column_names = ['replace_me', 'replace_me', 'replace_me']
sensitive_column_names = ['replace_me']

score = DisclosureProtection.compute(real_data = real.astype('object'), synthetic_data = synthetic_non_categorical, known_column_names = known_column_names, sensitive_column_names = sensitive_column_names)
print(f'Disclosure Protection Score: {score}')